# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saisathwik2703/flyrank.ai_internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
import os
import warnings
warnings.filterwarnings('ignore')

# Load data
if os.path.exists('data/raw/content_refresh_anonymized.csv'):
    df_raw = pd.read_csv('data/raw/content_refresh_anonymized.csv')
else:
    df_raw = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

df = df_raw.copy()

# ---- 1. Label ----
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# ---- 2. Log transforms for heavily skewed volume columns ----
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
            'pageviews_90d', 'users_90d', 'engaged_sessions_90d', 'scroll_events_90d']:
    df[f'log_{col}'] = np.log1p(df[col].fillna(0))

# ---- 3. Ratio features ----
df['click_per_impression'] = df['clicks_90d'] / df['impressions_90d'].clip(lower=1)
df['sessions_per_click'] = df['sessions_90d'] / df['clicks_90d'].clip(lower=1)
df['ai_session_ratio'] = df['ai_sessions_90d'] / df['sessions_90d'].clip(lower=1)
df['content_freshness_score'] = 1.0 / (df['days_since_last_update'] + 1)

# ---- 4. Missing value flags ----
df['has_keyword_data'] = (~df['search_volume'].isnull()).astype(int)
df['has_word_count'] = (~df['word_count'].isnull()).astype(int)

# ---- 5. Impute keyword columns by content_type median ----
for col in ['search_volume', 'competition', 'cpc']:
    medians = df.groupby('content_type')[col].transform('median')
    df[col] = df[col].fillna(medians).fillna(0)

# ---- 6. Impute length columns ----
for col in ['word_count', 'char_count']:
    medians = df.groupby('content_type')[col].transform('median')
    df[col] = df[col].fillna(medians).fillna(df[col].median())

# ---- 7. Categorical: fill with 'MISSING' then encode ----
CAT_COLS = ['competition_level', 'content_type', 'main_intent',
            'age_tier', 'freshness_tier', 'word_count_tier',
            'impression_tier', 'position_tier']
for col in CAT_COLS:
    df[col] = df[col].fillna('MISSING')

# Ordinal encode categoricals (NaN-safe since already filled)
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[CAT_COLS] = oe.fit_transform(df[CAT_COLS])

# ---- 8. Define final feature list ----
NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'click_per_impression', 'sessions_per_click', 'ai_session_ratio',
    'content_freshness_score',
    'has_keyword_data', 'has_word_count',
]

ALL_FEATURES = NUMERIC_FEATURES + CAT_COLS

X = df[ALL_FEATURES]
y = df['is_declining_label']

print(f'Feature matrix shape: {X.shape}')
print(f'Label distribution: {y.value_counts().to_dict()}')
print(f'\nAny remaining NaNs in X: {X.isnull().any().any()}')
print(f'\nFeature list ({len(ALL_FEATURES)} features):')
for f in ALL_FEATURES:
    print(f'  {f}')

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it is available at prediction time.*

| Feature | Meaning | Missing handling | Prediction-time? |
|---|---|---|---|
| `search_volume` | Keyword traffic potential | Median by content_type | Yes — from keyword tool |
| `competition` | Keyword difficulty 0–1 | Median by content_type | Yes |
| `cpc` | Keyword CPC | Median by content_type | Yes |
| `word_count` | Article length | Median by content_type | Yes — from CMS |
| `char_count` | Article char length | Same as word_count | Yes |
| `log_impressions_90d` | Log of 90d GSC impressions | log1p(0) = 0 | Yes — from GSC |
| `log_clicks_90d` | Log of 90d GSC clicks | log1p(0) = 0 | Yes |
| `log_sessions_90d` | Log of 90d GA4 sessions | log1p(0) = 0 | Yes |
| `log_ai_sessions_90d` | Log of 90d AI referral sessions | log1p(0) = 0 | Yes |
| `days_with_impressions` | Page consistency in search (0–90) | None expected | Yes |
| `days_with_sessions` | Page consistency in GA4 (0–90) | None expected | Yes |
| `content_age_days` | Days since page creation | None expected | Yes — from CMS |
| `days_since_last_update` | Days since last edit | None expected | Yes — from CMS |
| `ctr` | Click-through rate ×100 | None expected | Yes — GSC |
| `avg_position` | Average SERP position | None expected | Yes — GSC |
| `engagement_rate` | Engaged sessions / sessions ×100 | None expected | Yes — GA4 |
| `scroll_rate` | Scroll events / pageviews ×100 | None expected | Yes — GA4 |
| `ai_traffic_pct` | AI referral share ×100 | None expected | Yes — GA4 |
| `click_per_impression` | Effective CTR (raw ratio) | Denominator clipped at 1 | Yes |
| `sessions_per_click` | How many sessions per click | Denominator clipped at 1 | Yes |
| `ai_session_ratio` | AI sessions / total sessions | Denominator clipped at 1 | Yes |
| `content_freshness_score` | 1 / (days_since_last_update + 1) | Always computable | Yes |
| `has_keyword_data` | Flag: keyword context available | — | Yes |
| `has_word_count` | Flag: word count measured | — | Yes |
| `competition_level` | LOW/MEDIUM/HIGH (encoded) | Filled MISSING | Yes |
| `content_type` | Article type (encoded) | No missing | Yes |
| `main_intent` | Search intent (encoded) | Filled MISSING | Yes |
| `age_tier` | Binned content age (encoded) | No missing | Yes |
| `freshness_tier` | Binned recency (encoded) | No missing | Yes |
| `word_count_tier` | Binned length (encoded) | No missing | Yes |
| `impression_tier` | Binned traffic band (encoded) | No missing | Yes |
| `position_tier` | Binned ranking band (encoded) | No missing | Yes |

## 3. Leakage check: correlation of each feature with the label

*Any feature suspiciously correlated (>0.6 absolute) with `is_declining_label` is a leakage red flag.*

In [ ]:
# Compute Pearson correlation of each numeric feature with the label
corr = X[NUMERIC_FEATURES].corrwith(y).abs().sort_values(ascending=False)
print('Correlation of numeric features with is_declining_label:')
print(corr.round(3))

print('\nLeakage check: Any |corr| > 0.6?')
high_corr = corr[corr > 0.6]
if len(high_corr) > 0:
    print('WARNING — potential leakage:')
    print(high_corr)
else:
    print('No feature exceeds 0.6 absolute correlation. No leakage detected.')

# Verify the most dangerous forbidden features are excluded
FORBIDDEN = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d',
             'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
print('\nForbidden features present in X?')
for f in FORBIDDEN:
    present = f in X.columns
    status = 'FAIL — LEAKAGE' if present else 'OK — excluded'
    print(f'  {f}: {status}')

## 4. Privacy check

*Does the feature vector contain any identifying information about clients or end-users?*

**Privacy verdict: PASS — No identifying information in the feature vector.**

- `content_id` and `client_id` are **excluded from X** — they are used only for split construction and are pseudonymized.
- All volume metrics are 90-day aggregates — no individual user data.
- URL paths, page titles, and keyword text are not in the dataset at all.
- The data dictionary confirms all columns are aggregated metrics or pseudonymized IDs.
- `provider_used` and `model_used` are excluded (internal metadata, not model features).

**Data use compliance:** Per `DATA_USE.md`, the dataset is anonymized and may be used for model training within this internship scope. No columns shall be exported or shared outside the repo.